In [1]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
%matplotlib widget

from spectra.utilities import read_OE65PRO_spectrum_data_file, find_index_of_nearest

In [2]:
!jupyter lab --version

2.0.1


In [3]:
!conda list ipywidgets

# packages in environment at /Users/nordin/opt/miniconda3/envs/jupyter_py37:
#
# Name                    Version                   Build  Channel
ipywidgets                7.5.1                      py_0    conda-forge


In [4]:
!jupyter labextension list

JupyterLab v2.0.1
Known labextensions:
   app dir: /Users/nordin/opt/miniconda3/envs/jupyter_py37/share/jupyter/lab
        @jupyterlab/toc v3.0.0  enabled  OK
        jupyter-matplotlib v0.7.2  enabled  OK


In [5]:
!jupyter notebook --version

6.0.3


# Play with data

## Light source spectrum

In [6]:
lightsource_spectrometer_file = Path('raw_data') / 'QEPB00791_17-07-21-669.txt'

lightsource_spectrometer_header, lightsource_spectrometer_data = read_OE65PRO_spectrum_data_file(lightsource_spectrometer_file)
lightsource_spectrometer_data.shape

raw_data/QEPB00791_17-07-21-669.txt


(1044, 2)

In [7]:
# fig, ax = plt.subplots()
# ax.plot(lightsource_spectrometer_data[:,0], lightsource_spectrometer_data[:,1])

## PEGDA spectrum

In [8]:
PEGDA_spectrometer_file = Path('raw_data') / 'QEPB00791_17-10-07-458.txt'

PEGDA_spectrometer_header, PEGDA_spectrometer_data = read_OE65PRO_spectrum_data_file(PEGDA_spectrometer_file)
PEGDA_spectrometer_data.shape

raw_data/QEPB00791_17-10-07-458.txt


(1044, 2)

## Avobenzone in PEGDA spectrum

In [9]:
avobenzone_spectrometer_file = Path('raw_data') / 'QEPB00791_17-18-57-140.txt'

avobenzone_spectrometer_header, avobenzone_spectrometer_data = read_OE65PRO_spectrum_data_file(avobenzone_spectrometer_file)
avobenzone_spectrometer_data.shape

raw_data/QEPB00791_17-18-57-140.txt


(1044, 2)

## Plot

In [10]:
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(PEGDA_spectrometer_data[:,0], PEGDA_spectrometer_data[:,1], label='PEGDA')
ax.plot(avobenzone_spectrometer_data[:,0], avobenzone_spectrometer_data[:,1], label='avobenzone')
# ax.plot(lightsource_spectrometer_data[:,0], lightsource_spectrometer_data[:,1], label='Light Source')
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Signal')
ax.set_title("Raw Spectrum")
# ax.set_xlim([320, 460])
# ax.set_ylim([0, 60000])
ax.grid()
ax.legend();

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

# Restrict data sets to 300-540 nm

Reason:
- Low end - arbitrary
- High end - just between 532 nm laser wavelength and saturated spectrum values.

In [11]:
# Make sure both data sets are for the same wavelengths
assert np.allclose(PEGDA_spectrometer_data[:,0], avobenzone_spectrometer_data[:,0])

## Code

In [12]:
def find_indices_of_range(arr, low, high):
    assert len(arr.shape) == 1
    
    index_low = find_index_of_nearest(arr, low)
    index_high = find_index_of_nearest(arr, high)
    
    return index_low, index_high


def select_wavelength_range(data, wavelength_low, wavelength_high):
    assert len(data.shape) == 2
    assert data.shape[1] == 2
    
    index_low, index_high = find_indices_of_range(data[:,0], wavelength_low, wavelength_high)
    
    return data[index_low:index_high+1]


def calc_absorbance(source_spectrum, absorber_spectrum, wavelength_low, wavelength_high, force_positive=True, num_samples_for_average=10):
    
    temp_source = select_wavelength_range(source_spectrum, wavelength_low, wavelength_high)
    temp_absorber = select_wavelength_range(absorber_spectrum, wavelength_low, wavelength_high)
    
    absorbance = np.column_stack((temp_source[:,0], -np.log10(temp_absorber[:,1] / temp_source[:,1])))
    
    if force_positive:
        avg = np.mean(absorbance[-num_samples_for_average:, 1])
        if avg < 0.0:
            absorbance[:, 1] += -avg
        
    return absorbance

In [13]:
# Test select_wavelength_range()

temp = np.array([
    [1, 10],
    [2, 20],
    [3, 30],
    [4, 40],
    [5, 50],
    [6, 60],
])

temp2 = select_wavelength_range(temp, 2.1, 4.9)
temp2

array([[ 2, 20],
       [ 3, 30],
       [ 4, 40],
       [ 5, 50]])

In [14]:
# Test absorbance

temp1 = np.array([
    [1, 10],
    [2, 20],
    [3, 30],
    [4, 40],
    [5, 50],
    [6, 60],
])

temp2 = np.array([
    [1, 1],
    [2, 19],
    [3, 3],
    [4, 38],
    [5, 5],
    [6, 57],
])

calc_absorbance(temp1, temp2, 2.1, 4.9, False)

array([[2.        , 0.02227639],
       [3.        , 1.        ],
       [4.        , 0.02227639],
       [5.        , 1.        ]])

In [15]:
# Test absorbance

temp1 = np.array([
    [1, 10],
    [2, 20],
    [3, 30],
    [4, 40],
    [5, 50],
    [6, 60],
])

temp2 = np.array([
    [1, 1],
    [2, 22],
    [3, 3],
    [4, 44],
    [5, 55],
    [6, 66],
])

print(calc_absorbance(temp1, temp2, 2.1, 4.9, False))
print()
print(calc_absorbance(temp1, temp2, 2.1, 4.9, True, num_samples_for_average=2))

[[ 2.         -0.04139269]
 [ 3.          1.        ]
 [ 4.         -0.04139269]
 [ 5.         -0.04139269]]

[[2.         0.        ]
 [3.         1.04139269]
 [4.         0.        ]
 [5.         0.        ]]


# Calculate absorbance

In [16]:
wavelength_range = [300, 540]

absorbance = calc_absorbance(PEGDA_spectrometer_data, avobenzone_spectrometer_data, *wavelength_range)

In [17]:
avobenzone_spectrometer_header

['Data from QEPB00791_17-18-57-140.txt Node',
 '',
 'Date: Mon Jan 20 17:18:57 MST 2020',
 'User: ecestudent',
 'Spectrometer: QEPB0079',
 'Trigger mode: 0',
 'Integration Time (sec): 8.000000E-3',
 'Scans to average: 1',
 'Electric dark correction enabled: true',
 'Nonlinearity correction enabled: false',
 'Boxcar width: 0',
 'XAxis mode: Wavelengths',
 'Number of Pixels in Spectrum: 1044',
 '>>>>>Begin Spectral Data<<<<<']

In [18]:
output_data_file = Path(".") / "absorbance.csv"
np.savetxt(output_data_file, absorbance, delimiter=",", header="\n".join(avobenzone_spectrometer_header))

In [19]:
fig, ax = plt.subplots()
ax.plot(absorbance[:,0], absorbance[:,1])
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Signal')
ax.set_title("Avobenzone Absorbance")
ax.grid();

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

# Calculate molar absorptivity

##  Molar concentration

$$C_{molar} = \frac{C_{wwperc}}{100} \frac{\eta_{PEGDA}}{M_a}$$

$C_{molar}$ - Molar concentration, mol/L  
$C_{wwper}$ - Concentration w/w percent  
$\eta_{PEGDA}$ - Solvent density, g/L  
$M_a$ - Molar mass, g/mol  


In [20]:
def calc_molar_concentration(concentration_w_w_percent, density_of_solute_g_per_L, molar_mass_g_per_mole):
    """Calculates molar concentration in moles per Liter"""
    return (concentration_w_w_percent / 100.) * (density_of_solute_g_per_L / molar_mass_g_per_mole)

## Molar absorptivity

$$\epsilon(\lambda) = \frac{A(\lambda)}{C_{molar} l}$$

$\epsilon(\lambda)$ - Molar absorptivity, $\frac{\text{L}}{\text{mole}\:\text{cm}}$  
$A(\lambda)$ - Absorbance, unitless  
$C_{molar}$ - Molar concentration, mole/L  
$l$ - Length, cm  


In [21]:
def calc_molar_absorptivity(absorbance, molar_concentration, length_um):
    length_cm = 1.e-4 * length_um
    return absorbance / (molar_concentration * length_cm)

In [22]:
density_PEGDA = 1110  # g/L
molar_mass_avobenzone = 310.39  # g/mol
concentration_wwpercent_avobenzone = 0.24

molar_concentration_avobenzone = calc_molar_concentration(
                                    concentration_wwpercent_avobenzone, 
                                    density_PEGDA, 
                                    molar_mass_avobenzone)

print(molar_concentration_avobenzone)

0.00858275073294887


In [23]:
length_um = 60

molar_absorptivity = absorbance.copy()
molar_absorptivity[:, 1] = calc_molar_absorptivity(absorbance[:, 1], molar_concentration_avobenzone, length_um)

In [24]:
fig, ax = plt.subplots()
ax.plot(molar_absorptivity[:,0], molar_absorptivity[:,1])
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Molar absorptivity (L $\cdot$ mol$^{-1} \cdot$cm$^{-1}$)')
ax.set_title('Avobenzone, 01/20/2020 Measurement')
ax.grid();

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [25]:
output_data_file = Path(".") / "molar_absorptivity.csv"
np.savetxt(output_data_file, molar_absorptivity, delimiter=",", header="\n".join(avobenzone_spectrometer_header))

# Next

- Add data needed for molar absorptivity calculations, JSON?
- Put code in a utilities module and test
- Use code to analyze 2017 avobenzone absorption measurements and compare to 2020 measurements
- Do the same for Irgacure 819 photoinitiator as for avobenzone
- Document process of adding a photoinitiator or absorber